[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C55_TSR_Autonomous_Driving_Course/03_failure_modes/03_failure_modes.ipynb)

# 03 · 失效模式全景（四象限归因 / 物理下界 / 退化曲线 / 混淆对 / 优先级打分）

目标：把「TSR 哪里会错」从一份描述性清单，变成**一套能算出数字、能排出顺序、能落成工作项**的分析流程。
这正是 JD 里「Analyze TSR-related scenarios and failure cases」那条职责的日常形态。

本 notebook 你会亲手实现：

1. **四象限归因器**：把每个错误归入 漏检 / 误检 / 错分 / 定位不准，并按**像素尺寸分桶**
2. **物理下界计算器**：针孔模型 `p = f·S/Z` → 最远可检距离 → **留给决策的时间**
3. **退化影响曲线**：遮挡 / 运动模糊 / 低光 / 雾 对检测分数的影响，找出**临界退化强度**
4. **卷帘快门的量级核算**：证明「直觉高估」，并找出它真正危险的工况
5. **混淆矩阵分析**：找出最容易互相错分的标志对，并用**安全后果加权**重排（60→80 vs 60→40）
6. **车身贴纸抑制器**：跨模块关联，量化 FP 下降与误伤的取舍
7. **失效模式优先级打分器**：`F × S × D / C`，含「不可修」与「写进 ODD」的判定，以及预算约束下的选择

> 心智模型：**先分诊（可修吗？谁修？）→ 再分桶（集中在哪？）→ 再归因（修好能涨多少？）
> → 再排序（除以成本）→ 最后给闭环（触发器 + 数据动作 + 验证指标）。**

## 1 · 四象限归因器：把「错了」拆成四种不同的错

标准检测评测只给一个 mAP。要行动，必须知道**是哪一类错**。

In [ ]:
import numpy as np, math, json
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def attribute(gts, preds, iou_hi=0.5, iou_lo=0.1):
    '''四象限 + 细分归因。
       gts:   [(x1,y1,x2,y2,cls)]
       preds: [(x1,y1,x2,y2,cls,score)]  —— 按 score 降序贪心匹配
       返回 (counts, gt_status, pred_status)
       gt_status[i]   ∈ tp / cls_err / loc_err / both / fn
       pred_status[j] ∈ tp / cls_err / loc_err / both / fp_bkg'''
    order = sorted(range(len(preds)), key=lambda j: -preds[j][5])
    gt_status = ['fn'] * len(gts)
    pred_status = [None] * len(preds)
    used = [False] * len(gts)
    for j in order:
        p = preds[j]
        best, best_iou = -1, 0.0
        for i, g in enumerate(gts):
            if used[i]:
                continue
            v = iou_xyxy(p[:4], g[:4])
            if v > best_iou:
                best, best_iou = i, v
        if best < 0 or best_iou < iou_lo:
            pred_status[j] = 'fp_bkg'                      # ② 背景误检
            continue
        same = (p[4] == gts[best][4])
        if best_iou >= iou_hi:
            st = 'tp' if same else 'cls_err'               # ✅ 正确 / ③ 错分
        else:
            st = 'loc_err' if same else 'both'             # ④ 定位不准 / 两者都错
        pred_status[j] = st
        gt_status[best] = st
        used[best] = True
    keys = ['tp', 'cls_err', 'loc_err', 'both', 'fp_bkg', 'fn']
    counts = {k: 0 for k in keys}
    for s in pred_status:
        counts[s] += 1
    counts['fn'] = sum(1 for s in gt_status if s == 'fn')
    return counts, gt_status, pred_status

# 手工构造一个能逐项验算的小例子
gts = [(100, 100, 140, 140, 0),      # -> TP
       (200, 100, 240, 140, 1),      # -> 错分（框完全重合，类别错）
       (300, 100, 340, 140, 2),      # -> 定位不准（IoU=0.333，类别对）
       (400, 100, 440, 140, 3)]      # -> 漏检
preds = [(100, 100, 140, 140, 0, 0.90),
         (200, 100, 240, 140, 2, 0.80),
         (320, 100, 360, 140, 2, 0.70),
         (700, 700, 740, 740, 1, 0.60)]   # -> 背景误检

cnt, gst, pst = attribute(gts, preds)
print('四象限归因：', json.dumps(cnt, ensure_ascii=False))
print('GT   状态：', gst)
print('Pred 状态：', pst)
assert abs(iou_xyxy((320, 100, 360, 140), (300, 100, 340, 140)) - 1 / 3) < 1e-9
assert cnt == {'tp': 1, 'cls_err': 1, 'loc_err': 1, 'both': 0, 'fp_bkg': 1, 'fn': 1}, cnt
print()
print('✅ 同一份预测，拆开看是：1 正确 / 1 错分 / 1 定位不准 / 1 背景误检 / 1 漏检。')
print('   **这五个数字对应五种完全不同的修法**，而 mAP 只会给你一个数。')

### 按像素尺寸分桶：找出失效到底集中在哪

失效分析的第二步永远是**分桶**。TSR 最重要的分桶维度就是像素尺寸（≈ 距离）。

In [ ]:
def simulate_dataset(n_frames=3000, seed=1):
    '''合成一批「检测器输出」：
       · 检出概率随尺寸增大而上升（小目标漏检是主导失效）
       · 框误差的**绝对**像素量近似恒定 -> 小目标的相对误差更大 -> 定位不准更多
       · 分类正确率随尺寸上升（远处数字不可读）
       · 每帧有少量背景误检'''
    g = np.random.default_rng(seed)
    frames = []
    for _ in range(n_frames):
        gts, preds = [], []
        for _ in range(int(g.integers(1, 4))):
            s = float(np.clip(g.lognormal(mean=math.log(20), sigma=0.55), 7, 90))
            x, y = g.uniform(50, 1800), g.uniform(100, 600)
            cls = int(g.integers(0, 6))
            gts.append((x, y, x + s, y + s, cls))
            p_det = 1 / (1 + math.exp(-(s - 15.0) / 3.5))          # 尺寸->检出概率
            if g.random() < p_det:
                off = g.normal(0, 1.6, size=2)                      # 绝对像素误差 ~ 常数
                sc = 1 + g.normal(0, 0.09)
                s2 = s * sc
                px, py = x + off[0], y + off[1]
                p_cls = 1 / (1 + math.exp(-(s - 12.0) / 5.0))       # 尺寸->分类正确率
                c2 = cls if g.random() < p_cls else int(g.integers(0, 6))
                preds.append((px, py, px + s2, py + s2, c2, float(g.uniform(0.3, 1.0))))
        for _ in range(int(g.poisson(0.25))):                       # 背景误检
            x, y = g.uniform(0, 1800), g.uniform(0, 900)
            s = float(g.uniform(10, 40))
            preds.append((x, y, x + s, y + s, int(g.integers(0, 6)), float(g.uniform(0.3, 0.7))))
        frames.append((gts, preds))
    return frames

frames = simulate_dataset()
EDGES = [0, 12, 16, 24, 32, 48, 1e9]
LABEL = ['<12px', '12-16', '16-24', '24-32', '32-48', '>48px']

tot = {k: 0 for k in ['tp', 'cls_err', 'loc_err', 'both', 'fn']}
buck = {i: {k: 0 for k in tot} for i in range(len(LABEL))}
n_fp = 0
for gts, preds in frames:
    cnt, gst, pst = attribute(gts, preds)
    n_fp += cnt['fp_bkg']
    for g_, st in zip(gts, gst):
        s = g_[2] - g_[0]
        b = int(np.digitize(s, EDGES[1:-1]))
        buck[b][st] += 1
        tot[st] += 1

N = sum(tot.values())
print(f"{'尺寸桶':<9s}{'GT 数':>7s}{'漏检%':>8s}{'错分%':>8s}{'定位不准%':>10s}{'正确%':>8s}")
miss = {}
for b in range(len(LABEL)):
    n = sum(buck[b].values())
    if n == 0:
        continue
    miss[b] = buck[b]['fn'] / n
    print(f'{LABEL[b]:<9s}{n:>7d}{buck[b]["fn"] / n:>8.1%}'
          f'{buck[b]["cls_err"] / n:>8.1%}{(buck[b]["loc_err"] + buck[b]["both"]) / n:>10.1%}'
          f'{buck[b]["tp"] / n:>8.1%}')
print(f'\n整体：GT {N} 个，漏检 {tot["fn"] / N:.1%}，背景误检 {n_fp} 个'
      f'（{n_fp / len(frames):.3f} 个/帧）')

assert miss[0] > 0.5, '最小的桶漏检超过一半'
assert miss[5] < 0.02, '大目标几乎不漏'
assert all(miss[i] >= miss[i + 1] - 0.02 for i in range(5)), '漏检率随尺寸单调下降'
assert buck[0]['cls_err'] / max(1, sum(buck[0].values())) > \
       buck[5]['cls_err'] / max(1, sum(buck[5].values())), '小目标也更容易错分'

print()
print('✅ **Miss 在小尺寸桶里占绝对主导** —— 这就是「先做远距漏检」这个结论的来源。')
print('⚠️  整体漏检率 %.1f%% 这个数字本身没有行动价值：它混合了「不可修的物理下界」'
      % (100 * tot['fn'] / N))
print('    与「可修的模型问题」。**必须先分桶，再决定修哪个桶。**')

## 2 · 物理下界：先算出「哪些距离本来就不可能」

针孔模型 `p = f_px · S / Z`，`f_px = W / (2·tan(FOV/2))`。
**不先画出这条线，你会把精力浪费在信息论上不可能的任务上。**

In [ ]:
def focal_px(width_px, fov_deg):
    return width_px / (2.0 * math.tan(math.radians(fov_deg) / 2.0))

def sign_px(S, Z, f):
    '''标志在图像上的像素边长。S: 物理尺寸(m)，Z: 距离(m)。'''
    return f * S / Z

def max_range(S, p_min, f):
    '''要求至少 p_min 像素时，最远能看到多少米。'''
    return f * S / p_min

CAMS = [('广角 1920×1080 / FOV 60°', 1920, 60.0),
        ('长焦 1920×1080 / FOV 30°', 1920, 30.0)]
S_SIGN = 0.60                       # 限速牌直径 0.6 m

for nm, W, fov in CAMS:
    f = focal_px(W, fov)
    print(f'{nm}   f_px = {f:.1f}')
    print('   ' + ''.join(f'{f"{Z}m":>9s}' for Z in [20, 40, 60, 80, 100, 120]))
    print('   ' + ''.join(f'{sign_px(S_SIGN, Z, f):>9.1f}' for Z in [20, 40, 60, 80, 100, 120]))

f_wide = focal_px(1920, 60.0); f_tele = focal_px(1920, 30.0)
assert abs(sign_px(S_SIGN, 60, f_wide) - 16.6) < 0.3, '60 m 外约 17 px —— 记住这个数'
assert abs(max_range(S_SIGN, 16.0, f_wide) - 62.4) < 0.5
assert sign_px(S_SIGN, 60, f_tele) > 2.0 * sign_px(S_SIGN, 60, f_wide), '长焦把像素翻倍'

print(f'\n{"检出所需最小像素":>18s}' + ''.join(f'{f"{p}px":>12s}' for p in [10, 12, 16, 24]))
for nm, W, fov in CAMS:
    f = focal_px(W, fov)
    print(f'{nm[:6]:>18s}' + ''.join(f'{max_range(S_SIGN, p, f):>11.0f}m' for p in [10, 12, 16, 24]))
print()
print('✅ 「广角相机在 100 m 外只有 10 px」是物理事实，不是模型缺陷。')
print('✅ 想在更远处检出，**加长焦相机比换模型有效得多** —— 这是硬件层面的对策。')

In [ ]:
# 时间账：60 m 检出，留给系统多少时间？
def time_budget(Z_detect, v_kmh, confirm_frames=6, fps=30.0, Z_act=10.0):
    '''从检出到「必须完成动作」还剩多少秒；扣掉多帧确认的开销。'''
    v = v_kmh / 3.6
    t_total = (Z_detect - Z_act) / v
    t_confirm = confirm_frames / fps
    return t_total, t_confirm, t_total - t_confirm

print(f"{'车速':>7s}{'检出距离':>10s}{'总时间':>9s}{'多帧确认':>10s}{'留给规控':>10s}")
rows = {}
for v_kmh in [60, 90, 120]:
    for Zd in [40, 60, 80]:
        t, tc, left = time_budget(Zd, v_kmh)
        rows[(v_kmh, Zd)] = left
        flag = '   <- 危险' if left < 0.8 else ''
        print(f'{v_kmh:>5d}km/h{Zd:>9d}m{t:>8.2f}s{tc:>9.2f}s{left:>9.2f}s{flag}')

assert rows[(120, 40)] < rows[(120, 60)] < rows[(120, 80)]
assert rows[(120, 40)] < 1.0, '120 km/h 下 40 m 才检出，留给规控不到 1 秒'
assert rows[(60, 80)] > 3.0

print()
print('✅ **首次检出距离**是 TSR 最重要的工程指标之一（模块 05 会正式定义它）。')
print('⚠️  多帧确认（这里假设 6 帧 = 0.2 s）是稳定性的代价 ——')
print('    帧数越多越稳定但越晚，这个取舍在模块 04 会被量化。')
print('⚠️  注意 120 km/h + 40 m 检出这一格：留给规控 0.7 s。**这不是感知指标问题，')
print('    这是安全边界问题** —— 应当直接写进 ODD 与限速策略。')

## 3 · 退化影响曲线：遮挡 / 运动模糊 / 低光 / 雾

给检测分数造一个可控的代理（模板归一化相关 → sigmoid），
然后逐一施加物理退化，**找出每种退化的「临界强度」**（分数跌破工作阈值的那一点）。

In [ ]:
R = 48

def make_sign(R=48, seed=2):
    '''合成一块标志：外圈（红边）+ 内部图案。'''
    ax = np.arange(R); yy, xx = np.meshgrid(ax, ax, indexing='ij')
    c = (R - 1) / 2
    r = np.sqrt((yy - c) ** 2 + (xx - c) ** 2) / (R / 2)
    ring = ((r > 0.78) & (r <= 1.0)).astype(float)
    inner = (r <= 0.70).astype(float)
    g = np.random.default_rng(seed)
    pat = np.kron(g.normal(size=(4, 4)), np.ones((R // 4, R // 4)))
    img = 0.5 + 0.35 * ring + 0.25 * inner * pat
    return np.clip(img, 0.0, 1.0)

SIGN = make_sign(R)
TPL = (SIGN - SIGN.mean()) / (np.linalg.norm(SIGN - SIGN.mean()) + 1e-9)

def score(img):
    '''检测分数代理：与干净模板的归一化相关 -> sigmoid。'''
    z = img - img.mean()
    z = z / (np.linalg.norm(z) + 1e-9)
    ncc = float(np.dot(z.ravel(), TPL.ravel()))
    return 1.0 / (1.0 + math.exp(-12.0 * (ncc - 0.55)))

def occlude(img, ratio, g):
    '''随机矩形遮挡：覆盖 ratio 比例的面积。'''
    out = img.copy()
    if ratio <= 0:
        return out
    h = int(round(R * math.sqrt(ratio))); w = h
    y0 = int(g.integers(0, max(1, R - h))); x0 = int(g.integers(0, max(1, R - w)))
    out[y0:y0 + h, x0:x0 + w] = 0.15
    return out

def conv2d_same(img, k):
    kh, kw = k.shape; ph, pw = kh // 2, kw // 2
    pad = np.pad(img, ((ph, ph), (pw, pw)), mode='edge')
    out = np.zeros_like(img, dtype=float)
    for i in range(kh):
        for j in range(kw):
            out += k[i, j] * pad[i:i + img.shape[0], j:j + img.shape[1]]
    return out

def motion_kernel(length, angle_deg=15.0):
    '''有向线段核：长度 L = 相对速度 × 曝光时间（像素）。'''
    L = max(1, int(round(length)))
    k = np.zeros((2 * L + 1, 2 * L + 1))
    th = math.radians(angle_deg)
    for t in np.linspace(-L / 2.0, L / 2.0, 8 * L + 1):
        y = int(round(L + t * math.sin(th))); x = int(round(L + t * math.cos(th)))
        k[y, x] += 1.0
    return k / k.sum()

def lowlight(img, gain, g, read_noise=0.012):
    '''低光：整体压暗 + 光子噪声(泊松) + 读出噪声 + 8bit 量化。'''
    lam = np.clip(img * gain, 0, None) * 255.0
    photons = g.poisson(lam) / 255.0
    out = photons / max(gain, 1e-6) + g.normal(0, read_noise / max(gain, 1e-6), img.shape)
    return np.clip(np.round(out * 255) / 255.0, 0, 1)

def fog(img, beta, d=1.0, A=0.92):
    '''大气散射：I = J·t + A(1−t)，t = exp(−β d)。'''
    t = math.exp(-beta * d)
    return img * t + A * (1 - t)

g = np.random.default_rng(4)
print('干净分数 =', round(score(SIGN), 4))
assert score(SIGN) > 0.98

# **精确性质**：雾把对比度乘以 t —— 这是可以逐位验证的
for beta in [0.3, 0.8, 1.5]:
    t = math.exp(-beta * 1.0)
    assert abs(fog(SIGN, beta).std() - t * SIGN.std()) < 1e-12
print('✅ 验证大气散射模型：std(I) = t·std(J) 精确成立，t = exp(−βd)')
print('   ⇒ **对比度随距离指数衰减** —— 这就是远处标志在雾天先消失的原因。')

In [ ]:
def curve(fn, levels, n=40, seed=7):
    g = np.random.default_rng(seed)
    return [float(np.mean([score(fn(SIGN, lv, g)) for _ in range(n)])) for lv in levels]

OCC = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
BLUR = [0, 2, 4, 6, 9, 12, 16]
GAIN = [1.0, 0.5, 0.25, 0.12, 0.06, 0.03, 0.015]
BETA = [0.0, 0.3, 0.6, 1.0, 1.5, 2.2, 3.0]

c_occ = curve(lambda im, lv, gg: occlude(im, lv, gg), OCC)
c_blur = [float(score(conv2d_same(SIGN, motion_kernel(L)))) if L > 0 else score(SIGN) for L in BLUR]
c_low = curve(lambda im, lv, gg: lowlight(im, lv, gg), GAIN, n=12)
c_fog = [float(score(fog(SIGN, b))) for b in BETA]

def show(name, levels, scores, unit=''):
    print(f'{name:<14s}' + ''.join(f'{f"{lv}{unit}":>9s}' for lv in levels))
    print(f'{"  分数":<14s}' + ''.join(f'{s:>9.3f}' for s in scores))

show('遮挡比例', OCC, c_occ)
show('模糊核长', BLUR, c_blur, 'px')
show('光照增益', GAIN, c_low, '×')
show('雾 β', BETA, c_fog)

def critical(levels, scores, thr=0.5):
    '''分数首次跌破工作阈值的退化强度 —— 「临界退化强度」。'''
    for lv, s in zip(levels, scores):
        if s < thr:
            return lv
    return None

crit = {'遮挡': critical(OCC, c_occ), '模糊': critical(BLUR, c_blur),
        '低光': critical(GAIN, c_low), '雾': critical(BETA, c_fog)}
print('\n临界退化强度（分数跌破 0.5）:', crit)

assert c_occ[0] > c_occ[-1] and c_occ[0] > 0.9
assert c_blur[0] > c_blur[-1], '模糊越强分数越低'
assert c_fog[0] > c_fog[-1], '雾越浓分数越低'
assert crit['遮挡'] is not None and crit['遮挡'] <= 0.5, '遮挡超过一半基本必失效'
assert c_low[-1] < c_low[0], '光照越暗分数越低'

print()
print('✅ **临界强度**才是可以写进需求的东西：「遮挡 < 30%% 时必须检出」比')
print('   「提高鲁棒性」可验证得多，也可以直接变成一条回归门禁。')
print('⚠️  注意雾曲线的形状：前段平缓、后段陡降 —— 因为 sigmoid 的工作点在中段。')
print('    **这意味着「轻雾几乎无影响，浓雾断崖式失效」**，中间几乎没有过渡区。')

In [ ]:
# 卷帘快门：先算量级，再决定要不要管（直觉会严重高估它）
def rs_shift_px(h_px, T_readout=0.020, H=1080, v_img=0.0, f_px=1663.0, omega=0.0):
    '''目标自身跨越的行所对应的时间 dt = T·h/H；
       形变 = 横向像速 × dt   +   俯仰角速度引起的 f·ω·dt。'''
    dt = T_readout * (h_px / H)
    return abs(v_img) * dt + abs(f_px * omega * dt), dt

print(f"{'工况':<30s}{'牌高':>7s}{'dt(ms)':>9s}{'形变(px)':>10s}{'尺度误差':>10s}{'距离误差':>10s}")
CASES = [
    ('平稳行驶 · 横向 200 px/s', 16, 200.0, 0.0),
    ('平稳行驶 · 横向 800 px/s', 16, 800.0, 0.0),
    ('过减速带 · ω = 2 rad/s',   16, 200.0, 2.0),
    ('剧烈颠簸 · ω = 5 rad/s',   16, 200.0, 5.0),
    ('近距大牌 · ω = 2 rad/s',   120, 200.0, 2.0),
]
res = {}
for nm, h, v_img, om in CASES:
    d, dt = rs_shift_px(h, v_img=v_img, omega=om)
    rel = d / h
    res[nm] = rel
    print(f'{nm:<30s}{h:>6d}px{dt * 1000:>9.3f}{d:>10.3f}{rel:>9.1%}{rel:>10.1%}')

assert res['平稳行驶 · 横向 200 px/s'] < 0.01, '平稳行驶时卷帘快门对小牌的影响 < 1% —— 可忽略'
assert res['过减速带 · ω = 2 rad/s'] > 5 * res['平稳行驶 · 横向 200 px/s'], '颠簸才是主因'
assert res['剧烈颠簸 · ω = 5 rad/s'] > 0.10, '剧烈颠簸下尺度误差超过 10%'

print()
print('✅ Z = f·S/p 是**反比**关系 ⇒ 像素宽度错 x%%，距离就错约 x%%。')
print('✅ 结论（值得在面试里主动提）：**卷帘快门对小牌的横向形变可以忽略**')
print('   （因为一块 16 px 的牌只跨越 16 行，dt 只有全帧读出的 1.5%%）；')
print('   **真正危险的是颠簸/俯仰**，以及近距离跨越大量行的大牌，还有 LED 电子牌的相位干扰。')
print('⚠️  「先算量级再决定要不要管」比「记住卷帘快门有害」有用得多。')

## 4 · 混淆矩阵分析：找出最容易互相错分的标志对

错分不是均匀分布的，它**高度集中在少数几对**上。
更重要的是：**混淆的频次排序 ≠ 风险排序**（限速 60→80 与 60→40 频次相近，后果天差地别）。

In [ ]:
CLASSES = ['限速30', '限速40', '限速50', '限速60', '限速80', '限速100',
           '禁止驶入', '禁止左转', '禁止掉头', '停车让行', '注意行人', '注意学校']
NC = len(CLASSES)
LIMIT = {0: 30, 1: 40, 2: 50, 3: 60, 4: 80, 5: 100}       # 限速类的数值

def confusability():
    '''可混淆度矩阵 M[i][j]：越小越容易混。由「同组」+「数字形状相似」构成。'''
    M = np.full((NC, NC), 6.0)
    np.fill_diagonal(M, 0.0)
    speed, forbid, warn = list(range(6)), [6, 7, 8, 9], [10, 11]
    for grp, d in [(speed, 3.0), (forbid, 3.2), (warn, 2.6)]:
        for i in grp:
            for j in grp:
                if i != j:
                    M[i, j] = d
    # 数字形状相似：3/8、6/8、0/8 的笔画接近 -> 更容易混
    for i, j, d in [(3, 4, 1.6), (0, 4, 2.3), (1, 0, 2.4), (2, 1, 2.6), (4, 5, 2.5)]:
        M[i, j] = M[j, i] = d
    for i, j, d in [(7, 8, 2.0)]:                          # 禁止左转 / 禁止掉头
        M[i, j] = M[j, i] = d
    return M

M = confusability()

def simulate_confusion(M, n_per=4000, T=1.0, seed=3):
    g = np.random.default_rng(seed)
    cm = np.zeros((NC, NC), dtype=int)
    for c in range(NC):
        p = np.exp(-M[c] / T); p /= p.sum()
        cm[c] = np.bincount(g.choice(NC, size=n_per, p=p), minlength=NC)
    return cm

cm = simulate_confusion(M)
acc = np.trace(cm) / cm.sum()
print(f'整体准确率 = {acc:.4f}')
print(f"{'':<9s}" + ''.join(f'{c[-3:]:>7s}' for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f'{c:<9s}' + ''.join(f'{cm[i, j]:>7d}' for j in range(NC)))

def top_pairs(cm, k=6):
    '''按「双向混淆总数」排序的混淆对。'''
    out = []
    for i in range(NC):
        for j in range(i + 1, NC):
            out.append((int(cm[i, j] + cm[j, i]), i, j))
    return sorted(out, reverse=True)[:k]

print(f'\n{"最易混淆的标志对（按频次）":<28s}{"次数":>7s}')
tp = top_pairs(cm)
for n_, i, j in tp:
    print(f'{CLASSES[i] + " <-> " + CLASSES[j]:<28s}{n_:>7d}')

assert acc > 0.5, '大部分样本还是分对了'
assert (tp[0][1], tp[0][2]) == (3, 4), '最易混的应是「限速60 <-> 限速80」'
assert cm[3, 4] > cm[3, 6], '同组内的混淆远多于跨组'
print()
print('✅ 混淆**高度集中**：前 3 对就占了错分的很大一块 -> 定向补数据的性价比极高。')

In [ ]:
# 但频次不等于风险：把「安全后果」加权进去
def severity(i, j):
    '''把 i 错分成 j 的安全后果（1–10）。'''
    if i == j:
        return 0.0
    if i in LIMIT and j in LIMIT:
        # **判高了 = 超速 = 危险；判低了 = 保守 = 体验问题**
        return 9.0 if LIMIT[j] > LIMIT[i] else 2.5
    if i == 9:                    # 停车让行被判成别的 -> 冲入路口
        return 10.0
    if i in (6, 7, 8) and j not in (6, 7, 8):
        return 7.0                # 禁令被判成非禁令 -> 违规行为
    return 4.0

risk = np.zeros((NC, NC))
for i in range(NC):
    for j in range(NC):
        risk[i, j] = cm[i, j] * severity(i, j)

flat = sorted(((risk[i, j], i, j) for i in range(NC) for j in range(NC) if i != j), reverse=True)
print(f'{"最危险的错分方向（频次 × 后果）":<34s}{"次数":>7s}{"后果":>6s}{"风险":>9s}')
for r, i, j in flat[:8]:
    print(f'{CLASSES[i] + " -> " + CLASSES[j]:<34s}{cm[i, j]:>7d}{severity(i, j):>6.1f}{r:>9.0f}')

r_up = risk[3, 4]      # 限速60 -> 限速80（判高了，超速）
r_dn = risk[4, 3]      # 限速80 -> 限速60（判低了，保守）
print(f'\n限速60->80  次数 {cm[3, 4]:>5d}  风险 {r_up:>7.0f}')
print(f'限速80->60  次数 {cm[4, 3]:>5d}  风险 {r_dn:>7.0f}')

assert abs(cm[3, 4] - cm[4, 3]) / max(cm[3, 4], cm[4, 3]) < 0.15, '两个方向的频次相近（混淆是对称的）'
assert r_up > 3.0 * r_dn, '但风险差 3 倍以上 —— **方向不同，后果完全不同**'
assert flat[0][1] == 9 or severity(*flat[0][1:]) >= 9.0, '风险榜首必是高后果类'

print()
print('✅ **混淆矩阵必须看方向，而且必须加权。** 60->80 与 80->60 频次几乎相同，')
print('   但前者是超速（安全事故），后者只是过于保守（体验问题）。')
print('✅ 工程含义：① 评测要报**有向**的混淆而不是对称的「易混对」；')
print('   ② 分类器的决策可以**非对称**——在限速类上，判低比判高安全，')
print('      因此可以给「判高」施加额外的阈值代价（回到模块 02 的代价敏感决策）。')

## 5 · 误检源：用跨模块关联抑制「货车贴纸」

这是**低成本高收益**的典型：单靠 TSR 自己的像素信息几乎无法区分
「贴在卡车上的限速牌」与「路侧的限速牌」，但和车辆检测求一次交集就能解决。

In [ ]:
def contain_ratio(sign, veh):
    '''标志被车辆框包含的比例 = area(∩) / area(sign) —— 比 IoU 更合适。'''
    x1 = max(sign[0], veh[0]); y1 = max(sign[1], veh[1])
    x2 = min(sign[2], veh[2]); y2 = min(sign[3], veh[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a = (sign[2] - sign[0]) * (sign[3] - sign[1])
    return inter / a if a > 0 else 0.0

def simulate_sticker_scene(n=6000, seed=5):
    '''每帧：一辆前车 + 可能的车尾贴纸(应抑制) + 路侧真牌(不应抑制，但可能与车框在 2D 上重叠)。'''
    g = np.random.default_rng(seed)
    rows = []
    for _ in range(n):
        vw = g.uniform(200, 500); vh = vw * 0.8
        vx = g.uniform(600, 1100); vy = g.uniform(350, 500)
        veh = (vx, vy, vx + vw, vy + vh)
        if g.random() < 0.45:                              # 车尾贴纸：完全在车框内
            s = g.uniform(14, 34)
            sx = g.uniform(vx + 0.15 * vw, vx + 0.85 * vw - s)
            sy = g.uniform(vy + 0.45 * vh, vy + 0.9 * vh - s)
            rows.append(((sx, sy, sx + s, sy + s), veh, 1))
        if g.random() < 0.75:                              # 路侧真牌
            s = g.uniform(12, 40)
            # 多数在车框外；少部分因透视恰好**跨在车框边缘上** -> 这是「误伤」的来源
            if g.random() < 0.18:
                sx = vx - s * g.uniform(0.15, 0.85)         # 横跨车框左边界，部分重叠
                sy = g.uniform(vy, vy + vh - s)
            else:
                sx = g.choice([g.uniform(50, max(60, vx - 60)), g.uniform(vx + vw + 40, 1850)])
                sy = g.uniform(120, 420)
            rows.append(((sx, sy, sx + s, sy + s), veh, 0))
    return rows

scene = simulate_sticker_scene()
n_sticker = sum(r[2] for r in scene); n_real = len(scene) - n_sticker
KM = 3000.0                                                 # 假设这批帧来自约 3000 km 的车队数据
print(f'样本：车身贴纸 {n_sticker} 个，路侧真牌 {n_real} 个；假设来自 {KM:.0f} km 车队数据')
print(f"{'抑制阈值':>9s}{'贴纸抑制率':>12s}{'真牌误伤率':>12s}{'残余 FP/km':>13s}{'真牌损失/km':>13s}")
best = None
for thr in [0.30, 0.50, 0.70, 0.85, 0.95, 1.01]:
    sup_bad = sum(1 for s, v, lab in scene if lab == 1 and contain_ratio(s, v) >= thr)
    sup_good = sum(1 for s, v, lab in scene if lab == 0 and contain_ratio(s, v) >= thr)
    fp_km = (n_sticker - sup_bad) / KM
    loss_km = sup_good / KM
    print(f'{thr:>9.2f}{sup_bad / n_sticker:>12.1%}{sup_good / n_real:>12.1%}'
          f'{fp_km:>13.3f}{loss_km:>13.3f}')
    if thr <= 1.0 and (best is None or (fp_km + 3 * loss_km) < best[1]):
        best = (thr, fp_km + 3 * loss_km)

base_fp = n_sticker / KM
sup95 = sum(1 for s, v, lab in scene if lab == 1 and contain_ratio(s, v) >= 0.95)
hurt95 = sum(1 for s, v, lab in scene if lab == 0 and contain_ratio(s, v) >= 0.95)
print(f'\n不做抑制：FP/km = {base_fp:.3f}；阈值 0.95 抑制后 = {(n_sticker - sup95) / KM:.3f}')
assert sup95 / n_sticker > 0.95, '完全包含判据能抑制绝大多数车身贴纸'
assert hurt95 / n_real < 0.03, '误伤率很低（真牌很少被车框完全包住）'
assert (n_sticker - sup95) / KM < base_fp * 0.1, 'FP/km 下降一个数量级'
print(f'最优工作点（代价 FP + 3×误伤）：阈值 = {best[0]:.2f}')

print()
print('✅ 一条跨模块规则把这类 FP 降了一个数量级，**成本几乎为零**（不需要重训任何模型）。')
print('✅ 这就是优先级表里「货车贴纸误检」排在最前面的原因：**低成本 × 中高收益**。')
print('⚠️  用「包含比例」而不是 IoU：贴纸远小于车框，IoU 永远很小，用 IoU 会完全失效。')
print('   **度量选错，一个正确的想法也会失败** —— 这类细节是工程与 demo 的分界线。')

## 6 · 失效模式优先级打分器

`Priority = F × S × D / C`：频率 × 安全后果 × 静默性 ÷ 修复成本。
**「除以成本」和「把不可修的剔除」这两步，是它和拍脑袋排序的分界。**

In [ ]:
FAILURES = [
    # name,                    F 频率, S 后果, D 静默性, C 成本(人月), fixable, note
    ('远距漏检 40-70m',            8, 9, 3, 3.0, True,  '可修区间'),
    ('远距漏检 >90m',              9, 4, 3, 99.0, False, '物理下界，不可修'),
    ('货车贴纸误检',                5, 6, 1, 0.5, True,  '跨模块关联即可'),
    ('限速数字混淆 60/80',          5, 9, 3, 1.0, True,  '集中在少数相似对'),
    ('对向车道误采纳',              9, 9, 3, 8.0, True,  'mAP 看不见，架构性工作'),
    ('隧道口过渡帧',                3, 6, 2, 8.0, True,  '需 ISP 配合'),
    ('夜间眩光漏检',                4, 7, 3, 4.0, True,  '真采 + 合成'),
    ('雨雾对比度下降',              4, 6, 2, 2.0, True,  '物理可合成'),
    ('积雪覆盖牌面',                1, 6, 2, 10.0, True,  '地域性极低频'),
    ('电子牌频闪缺笔画',            2, 7, 3, 5.0, True,  '曝光与刷新同步'),
    ('临时施工牌漏检',              2, 8, 3, 4.0, True,  '地图先验此时是错的'),
]

def priority(f):
    return f[1] * f[2] * f[3] / f[4]

rank = sorted([f for f in FAILURES if f[5]], key=priority, reverse=True)
print(f"{'失效模式':<20s}{'F':>3s}{'S':>3s}{'D':>3s}{'C':>6s}{'优先级':>9s}  {'星级':<7s}备注")
for f in rank:
    p = priority(f)
    stars = '★' * min(5, max(1, int(round(p / max(priority(x) for x in rank) * 5))))
    print(f'{f[0]:<20s}{f[1]:>3d}{f[2]:>3d}{f[3]:>3d}{f[4]:>6.1f}{p:>9.1f}  {stars:<7s}{f[6]}')

dropped = [f[0] for f in FAILURES if not f[5]]
print(f'\n已剔除（不可修，不进 backlog）：{dropped}')

names = [f[0] for f in rank]
assert '远距漏检 >90m' not in names, '物理下界必须先被剔除'
assert names.index('货车贴纸误检') < 3, '低成本高收益应排在最前'
assert names.index('积雪覆盖牌面') >= len(names) - 3, '极低频 + 高成本 -> 垫底（写进 ODD）'
assert priority(('x', 5, 9, 3, 1.0)) > priority(('x', 5, 9, 3, 8.0)), '成本越高优先级越低'

print()
print('✅ 「积雪覆盖」的正确处理不是修，而是**写进 ODD 并确保降级行为安全**。')
print('✅ D（静默性）在分子上：**越难被发现的失效，其真实代价被系统性低估**，')
print('   所以要额外加权。漏检、错误关联都属于这一类。')

In [ ]:
# 预算约束下的选择：按风险 vs 贪心 ROI vs 0/1 背包最优
def risk(f):
    return f[1] * f[2] * f[3]            # 未除以成本的绝对风险

def greedy(failures, budget, key):
    cand = sorted([f for f in failures if f[5]], key=key, reverse=True)
    chosen, spent, gained = [], 0.0, 0.0
    for f in cand:
        if spent + f[4] <= budget + 1e-9:
            chosen.append(f[0]); spent += f[4]; gained += risk(f)
    return chosen, spent, gained

def select_optimal(failures, budget, unit=0.5):
    '''0/1 背包 DP：在预算内最大化风险削减总量。'''
    cand = [f for f in failures if f[5]]
    W = int(round(budget / unit))
    w = [int(round(f[4] / unit)) for f in cand]
    v = [risk(f) for f in cand]
    dp = [0] * (W + 1)
    keep = [[] for _ in range(W + 1)]
    for i in range(len(cand)):
        for cap in range(W, w[i] - 1, -1):
            if dp[cap - w[i]] + v[i] > dp[cap]:
                dp[cap] = dp[cap - w[i]] + v[i]
                keep[cap] = keep[cap - w[i]] + [cand[i][0]]
    names_ = keep[W]
    spent = sum(f[4] for f in cand if f[0] in names_)
    return names_, spent, float(dp[W])

BUDGET = 8.0
c_risk, s_risk, g_risk = greedy(FAILURES, BUDGET, risk)         # ❌ 忽略成本
c_roi, s_roi, g_roi = greedy(FAILURES, BUDGET, priority)        # ✅ 按 ROI 贪心
c_opt, s_opt, g_opt = select_optimal(FAILURES, BUDGET)          # ✅✅ 0/1 背包最优

print(f'预算 {BUDGET} 人月')
for tag, c, s_, g_ in [('① 按绝对风险排（常见错误）', c_risk, s_risk, g_risk),
                       ('② 按 ROI = 风险/成本 贪心', c_roi, s_roi, g_roi),
                       ('③ 0/1 背包最优解', c_opt, s_opt, g_opt)]:
    print(f'\n{tag}：花费 {s_:.1f} 人月，削减风险 {g_:.0f}')
    for nm in c:
        print(f'     · {nm}')

assert g_roi > g_risk, '同样预算下，按 ROI 选远优于按绝对风险选'
assert g_opt >= g_roi, '贪心 ROI 只是近似；0/1 背包才是最优'
assert len(c_roi) > len(c_risk), '忽略成本会一头扎进一个昂贵项目，把预算吃光'
assert '货车贴纸误检' in c_roi and '货车贴纸误检' not in c_risk

print(f'\n✅ 同样 {BUDGET} 人月：按风险选削减 {g_risk:.0f}，按 ROI 选削减 {g_roi:.0f}'
      f'（+{g_roi / g_risk - 1:.0%}），最优解 {g_opt:.0f}。')
print('✅ **「按风险从高到低做」是最常见也最昂贵的排序错误** ——')
print('   它会让团队一头扎进一个高风险但极贵的问题（这里是「对向车道误采纳」，')
print('   8 人月吃光全部预算），而放着三四个便宜的修复不做。')
print('⚠️  但也别迷信贪心 ROI：它对 0/1 选择只是近似（这里 %.0f vs 最优 %.0f）。'
      % (g_roi, g_opt))
print('   **真正的排序问题是一个背包问题**，规模小的时候直接 DP 求最优即可。')

## ✏️ 练习 1：可修漏检报告

失效分析的第一步是**分诊**：把「不可修的物理下界」从统计里剔除，否则所有优先级都会失真。

实现 `fixable_miss_report(gt_sizes, statuses, edges, min_px)`：

- `gt_sizes`：每个 GT 的像素边长；`statuses`：对应的 `attribute()` 输出的 gt_status
- `edges`：分桶边界（左闭右开，首尾为 0 与 ∞）；`min_px`：物理可检下界（小于它的漏检视为不可修）
- 返回 `{'buckets': [(label_idx, n, n_miss, miss_rate)], 'fixable_miss': …, 'unfixable_miss': …, 'worst': label_idx}`
- `worst` = **可修漏检数量最多**的桶（不是漏检率最高的桶——率最高的往往是不可修的那个）

In [ ]:
def fixable_miss_report(gt_sizes, statuses, edges, min_px):
    # TODO: ① 按 edges 分桶统计 n / n_miss / miss_rate
    #       ② 把 size < min_px 的漏检记为 unfixable，其余记为 fixable
    #       ③ worst = 可修漏检**数量**最多的桶下标
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
sizes, sts = [], []
for gts, preds in frames:
    _, gst, _ = attribute(gts, preds)
    for g_, st in zip(gts, gst):
        sizes.append(g_[2] - g_[0]); sts.append(st)

rep = fixable_miss_report(sizes, sts, EDGES, min_px=12.0)
print(f"{'桶':<9s}{'GT':>7s}{'漏检':>7s}{'漏检率':>9s}")
for b, n_, nm_, mr in rep['buckets']:
    print(f'{LABEL[b]:<9s}{n_:>7d}{nm_:>7d}{mr:>9.1%}')
print(f"\n可修漏检 {rep['fixable_miss']}  |  不可修（<12px）{rep['unfixable_miss']}"
      f"  |  最该修的桶 = {LABEL[rep['worst']]}")

assert rep['fixable_miss'] + rep['unfixable_miss'] == sum(1 for s in sts if s == 'fn')
assert rep['unfixable_miss'] > 0, '一定有一批漏检落在物理下界之下'
assert rep['worst'] != 0, '**漏检率最高的桶是 <12px，但它不可修** —— worst 不应指向它'
assert rep['buckets'][0][3] > rep['buckets'][-1][3], '漏检率随尺寸下降'
tiny = rep['buckets'][0]
assert tiny[2] > 0 and rep['fixable_miss'] < sum(1 for s in sts if s == 'fn')
print('✅ 练习 1 通过：**先剔除不可修的部分，再决定修哪个桶** —— 这一步不做，')
print('   「整体漏检率 18%」这种数字会把团队引向一个根本无法完成的任务。')

## ✏️ 练习 2：反推相机配置

「要在 80 m 检出限速牌」是一个产品需求。**把它翻译成相机参数，是感知工程师的基本功。**

实现 `plan_camera(S, p_min, Z_target, width_px)`，返回 `{'f_px':…, 'fov_deg':…}`：
使得物理尺寸 `S` 的标志在距离 `Z_target` 处至少占 `p_min` 像素。

提示：`f = p_min · Z / S`，`FOV = 2·arctan(W / (2f))`。

In [ ]:
def plan_camera(S, p_min, Z_target, width_px):
    # TODO: 由 p = f·S/Z 反解 f，再由 f 反解 FOV
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
r80 = plan_camera(0.60, 16.0, 80.0, 1920)
r60 = plan_camera(0.60, 16.0, 60.0, 1920)
assert abs(r80['f_px'] - 2133.33) < 0.1, 'f = 16 × 80 / 0.6'
assert abs(r80['fov_deg'] - 48.46) < 0.2, r80
assert r60['fov_deg'] > r80['fov_deg'], '要求的距离越近，允许的 FOV 越宽'
assert abs(sign_px(0.60, 80.0, r80['f_px']) - 16.0) < 1e-6, '闭环校验：反推出来的配置确实给出 16 px'

print(f"{'目标检出距离':>12s}{'所需 f_px':>11s}{'所需 FOV':>10s}{'与广角 60° 的比较'}")
for Z in [40, 60, 80, 100]:
    r = plan_camera(0.60, 16.0, float(Z), 1920)
    cmp = '广角够用' if r['fov_deg'] >= 60.0 else f'需长焦（窄 {60 - r["fov_deg"]:.0f}°）'
    print(f'{Z:>11d}m{r["f_px"]:>11.0f}{r["fov_deg"]:>9.1f}°   {cmp}')
print('✅ 练习 2 通过：**产品需求 -> 相机参数**的翻译只有两行公式，')
print('   但它能在项目最早期就否掉「用一个广角相机做全部 TSR」这种方案。')

## ✏️ 练习 3：有向风险混淆表

**混淆的频次排序 ≠ 风险排序**，而且混淆是**有方向**的。

实现 `risky_confusions(cm, severity_fn, k)`：返回按 `cm[i][j] × severity(i,j)` 降序的
前 `k` 个 `(i, j, count, sev, risk)`（跳过 `i == j`）。
再用它证明：**两个排序会发生反转** —— 存在一对混淆 A、B，A 的频次高于 B，但风险低于 B。

In [ ]:
def risky_confusions(cm, severity_fn, k=5):
    # TODO: 枚举所有 i != j，按 count × severity 降序返回前 k 个五元组
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
top_risk = risky_confusions(cm, severity, k=5)
by_count = sorted(((int(cm[i, j]), i, j) for i in range(NC) for j in range(NC) if i != j),
                  reverse=True)

print(f"{'按风险排':<24s}{'次数':>7s}{'后果':>6s}{'风险':>9s}")
for i, j, c_, sv, rk in top_risk:
    print(f'{CLASSES[i] + " -> " + CLASSES[j]:<24s}{c_:>7d}{sv:>6.1f}{rk:>9.0f}')
rank_count = [(i, j) for _, i, j in by_count]
rank_risk = [(i, j) for i, j, _, _, _ in risky_confusions(cm, severity, k=NC * NC)]
A, B = (8, 7), (0, 4)          # 禁止掉头->禁止左转（高频低后果） vs 限速30->限速80（低频高后果）
print(f'\n{"":<24s}{"频次名次":>10s}{"风险名次":>10s}{"次数":>7s}{"后果":>6s}')
for pr in [A, B]:
    print(f'{CLASSES[pr[0]] + " -> " + CLASSES[pr[1]]:<24s}'
          f'{rank_count.index(pr) + 1:>10d}{rank_risk.index(pr) + 1:>10d}'
          f'{int(cm[pr]):>7d}{severity(*pr):>6.1f}')

assert len(top_risk) == 5 and all(len(t) == 5 for t in top_risk)
assert top_risk[0][4] >= top_risk[1][4] >= top_risk[2][4], '必须按风险降序'
assert all(t[0] != t[1] for t in top_risk), '不能包含对角线'
assert abs(top_risk[0][4] - cm[top_risk[0][0], top_risk[0][1]] * severity(*top_risk[0][:2])) < 1e-9
# 关键结论：两个排序会**反转**
assert rank_count.index(A) < rank_count.index(B), '按频次：A 排在 B 前面'
assert rank_risk.index(A) > rank_risk.index(B), '按风险：B 反超 A —— **排序反转**'
print('✅ 练习 3 通过：A 的次数更多，但 B 的风险更高 —— **只看频次会把资源投错地方**。')
print('   评测要报**有向、加权**的混淆，而不是对称的「易混对」列表。')

## ✏️ 练习 4：预算-收益曲线（还值不值得再投）

有了打分器，下一个问题是「**该投多少人月**」。
对一系列预算求最优风险削减，画出边际收益曲线，就能回答它。

实现 `budget_curve(failures, budgets)`：对每个预算调用 `select_optimal`，返回
`[(budget, gained, marginal)]`，其中 `marginal = (gained_i − gained_{i−1}) / (budget_i − budget_{i−1})`
（第一项的 marginal = `gained_0 / budget_0`）。

In [ ]:
def budget_curve(failures, budgets):
    # TODO: 对每个 budget 调用 select_optimal，算累计收益与相邻两点之间的边际收益
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
BUDGETS = [1.0, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 16.0, 24.0, 32.0]
curve_ = budget_curve(FAILURES, BUDGETS)
print(f"{'预算(人月)':>10s}{'累计风险削减':>14s}{'边际收益/人月':>15s}")
for b, g_, mg in curve_:
    print(f'{b:>10.1f}{g_:>14.0f}{mg:>15.1f}')

gains = [g_ for _, g_, _ in curve_]
assert all(gains[i] <= gains[i + 1] + 1e-9 for i in range(len(gains) - 1)), '预算越多收益不会下降'
assert curve_[0][2] > curve_[-1][2], '**边际收益整体递减** —— 后面的钱越来越不值'
assert abs(curve_[0][1] - curve_[0][2] * BUDGETS[0]) < 1e-6, '第一点的 marginal = gained / budget'
assert gains[-1] > gains[0] * 3

half = next(b for b, g_, _ in curve_ if g_ >= 0.8 * gains[-1])
print(f'\n用 {half:.0f} 人月即可拿到最终收益的 80%%（总共 {BUDGETS[-1]:.0f} 人月）。')
print('✅ 练习 4 通过：**边际收益曲线是「还值不值得再投」的唯一可信依据。**')
print('   同样的曲线在 C58 会以「还值不值得再标数据」的形式再出现一次。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def fixable_miss_report(gt_sizes, statuses, edges, min_px):
    nb = len(edges) - 1
    n = [0] * nb; nmiss = [0] * nb; fixable = [0] * nb
    unfix = 0
    for s, st in zip(gt_sizes, statuses):
        b = int(np.digitize(s, edges[1:-1]))
        n[b] += 1
        if st == 'fn':
            nmiss[b] += 1
            if s < min_px:
                unfix += 1
            else:
                fixable[b] += 1
    buckets = [(b, n[b], nmiss[b], (nmiss[b] / n[b] if n[b] else 0.0))
               for b in range(nb) if n[b] > 0]
    return {'buckets': buckets, 'fixable_miss': sum(fixable),
            'unfixable_miss': unfix, 'worst': int(np.argmax(fixable))}

In [ ]:
# 练习 2 参考答案
def plan_camera(S, p_min, Z_target, width_px):
    f = p_min * Z_target / S                                   # 由 p = f·S/Z 反解
    fov = 2.0 * math.degrees(math.atan(width_px / (2.0 * f)))  # 由 f 反解 FOV
    return {'f_px': f, 'fov_deg': fov}

In [ ]:
# 练习 3 参考答案
def risky_confusions(cm, severity_fn, k=5):
    out = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i == j:
                continue
            sv = severity_fn(i, j)
            out.append((i, j, int(cm[i, j]), float(sv), float(cm[i, j] * sv)))
    return sorted(out, key=lambda t: -t[4])[:k]

In [ ]:
# 练习 4 参考答案
def budget_curve(failures, budgets):
    out, prev_b, prev_g = [], 0.0, 0.0
    for b in budgets:
        _, _, g_ = select_optimal(failures, b)
        out.append((b, g_, (g_ - prev_g) / (b - prev_b)))
        prev_b, prev_g = b, g_
    return out

---
## 🧪 真实工程胶囊：失效模式台账 + 触发器 + 场景化回归门禁

可以原样复制进真实项目。三样东西缺一不可：
**一份带 ODD 边界的失效台账**、**每类失效的线上触发器**、**一套能挡住回退的场景化门禁**。

In [ ]:
RECIPE = r'''
# ─────────────────────────────────────────────────────────────
# ① 失效模式台账（configs/tsr_failure_modes.yaml）—— 评审用的唯一事实来源
# ─────────────────────────────────────────────────────────────
failure_modes:
  - id: FM-001
    name: far_range_miss_40_70m
    quadrant: FN                    # FN / FP / CLS / LOC
    cause_layer: [model, data]      # sensor / isp / model / data / association
    F: 8            # 频率  1-10（用车队日志估，不要拍脑袋）
    S: 9            # 安全后果 1-10
    D: 3            # 静默性 1-3（3 = 线上完全观测不到，必须靠回归集）
    C: 3.0          # 修复成本（人月）
    fixable: true
    odd_note: null
  - id: FM-002
    name: truck_sticker_fp
    quadrant: FP
    cause_layer: [association]
    F: 5, S: 6, D: 1, C: 0.5, fixable: true
  - id: FM-009
    name: snow_covered_sign
    quadrant: FN
    cause_layer: [sensor]
    F: 1, S: 6, D: 2, C: 10.0
    fixable: true
    odd_note: "积雪覆盖牌面时系统降级并提示接管；不承诺识别"   # <- 写进 ODD
  - id: FM-000
    name: beyond_90m_miss
    fixable: false
    reason: "物理下界：1920/60° 相机在 90m 外 0.6m 标志 < 11px"   # <- 不进 backlog

# 优先级 = F × S × D / C；不可修的**先剔除**再排序
# 预算分配是一个 0/1 背包问题，规模小时直接 DP 求最优

# ─────────────────────────────────────────────────────────────
# ② 线上触发器（每一类失效配一个，否则台账只是一张纸）
# ─────────────────────────────────────────────────────────────
TRIGGERS = {
  # 远距漏检：近处已确认的 track，回放它最早在哪一帧被检出
  "far_range_miss": lambda tr: tr.confirmed and tr.first_det_px < 14,
  # 货车贴纸：标志框被车辆框「包含」（用 containment 而不是 IoU！）
  "truck_sticker":  lambda d, vehs: any(contain_ratio(d.box, v.box) > 0.9 for v in vehs),
  # 数字混淆：多帧类别在同组内跳变，或 top1-top2 margin 过小
  "digit_confuse":  lambda tr: tr.class_switches >= 2 or tr.margin < 0.15,
  # 对向车道误采纳：上报的约束与地图/导航不一致
  "wrong_lane":     lambda out, hd: out.speed_limit != hd.lane_speed_limit,
  # 隧道/夜间：场景标签 + 分数骤降
  "scene_drop":     lambda f: f.scene in ("tunnel_in", "tunnel_out", "night_glare")
                              and f.mean_score < 0.55 * f.baseline_score,
}
# 触发率必须可控（回传带宽有限）：给每个触发器做 PR 分析，见 C58 模块 03

# ─────────────────────────────────────────────────────────────
# ③ 场景化回归门禁（CI 里跑，任何一条不过就不允许发布）
# ─────────────────────────────────────────────────────────────
GATES = [
  # (切片,                       指标,               门槛,   容差)
  ("size_16_24px",              "recall",           0.82,  -0.005),   # 不许回退
  ("size_lt_12px",              "recall",           None,   None),    # 物理下界，不设门槛
  ("night",                     "recall",           0.75,  -0.010),
  ("tunnel_transition",         "recall",           0.60,  -0.020),
  ("rain_fog",                  "recall",           0.78,  -0.010),
  ("occlusion_25_50",           "recall",           0.70,  -0.015),
  ("all",                       "FP_per_km",        0.30,  +0.02),    # 越低越好
  ("speed_limit_pairs",         "risky_confusion",  0.010, +0.002),   # 有向、加权
  ("adopted_constraints",       "adoption_acc",     0.95,  -0.005),   # 「该不该采纳」
]
# 铁律：**每个失效模式都必须对应至少一条门禁**，否则修好了也会悄悄退回去。

# ─────────────────────────────────────────────────────────────
# ④ badcase 分诊清单（每个 badcase 花 30 秒，先分堆再深挖）
# ─────────────────────────────────────────────────────────────
# 1. 四象限：漏检 / 误检 / 错分 / 定位不准？
# 2. 人眼可读性：原图上人能认出来吗？
#      不能 + 像素饱和/接近全黑  -> 传感器/ISP 问题（不是模型问题）
#      不能 + 像素太少          -> 物理下界（不可修，别进 backlog）
#      能                       -> 模型/数据问题（可修）
# 3. 是不是「认对了但不该采纳」？-> 关联问题，mAP 看不见，要单独立项
# 4. 时序上其他帧能看清吗？      -> 时序可救 -> 优先做多帧融合而不是单帧鲁棒性
'''
print(RECIPE)
for key in ['failure_modes', 'fixable: false', 'odd_note', 'TRIGGERS', 'GATES',
            'containment', 'adoption_acc', '人眼可读性']:
    assert key in RECIPE, key
print('✅ 配方覆盖：失效台账（含 ODD 与不可修剔除）/ 线上触发器 / 场景化门禁 / badcase 分诊清单')

### 小结

- **先把「错了」拆成四象限**：漏检 / 误检 / 错分 / 定位不准。四类的成因、代价、修法几乎没有交集，
  混在一个 mAP 里谈必然得不出行动项。**漏检是静默失败**，只能靠主动构造的回归集监控。
- **先算物理下界再谈模型**。`p = f·S/Z`：1920/60° 相机在 60 m 外只有约 17 px，100 m 外只有 10 px。
  **不可修的失败不该进 backlog**；想看更远，加长焦相机比换模型有效得多。
- **分桶是失效分析的第二步**。TSR 的漏检高度集中在小尺寸桶；整体漏检率这个数字本身没有行动价值。
- **能写出物理模型的退化（雾、模糊、低光、卷帘快门）都该用合成增强覆盖**；
  写不出模型的（积雪、涂鸦、褪色）只能真采。这条判据可以直接用来分配数据预算。
- **临界退化强度**（分数跌破工作阈值的那一点）才是能写进需求与门禁的东西。
- **卷帘快门的直觉会严重高估它**：16 px 的牌只跨越 16 行，横向形变 < 1%；
  真正危险的是**颠簸/俯仰**（尺度误差直接变成同比例的距离误差）与 LED 相位干扰。
- **混淆必须看方向并加权**：限速 60→80 与 80→60 频次相同，风险差 3 倍以上。
  由此推出：**分类器的决策可以非对称**（在限速类上给「判高」加额外阈值代价）。
- **很多「TSR 失效」不是识别错误，而是语义与关联错误**——货车贴纸、对向车道、辅路牌。
  它们在 mAP 上完全不可见，必须有「采纳正确率」这一层指标，并把关联做成显式可测的模块。
- **优先级 = F × S × D / C**，先剔除不可修的，把极低频高成本的写进 **ODD**。
  **「按风险从高到低做」是最常见也最昂贵的排序错误**；预算分配本质是一个背包问题。
- **每个失效模式都要配一个触发器 + 一个数据动作 + 一条门禁**，否则修好了也会悄悄退回去。

下一站：**模块 04 · 时序与多帧融合** —— 本模块反复出现的「时序可救 / 时序救不了」，
到那里会变成可以量化的跟踪、投票与迟滞设计。